In [ ]:
# All imports + the LLM client used in this notebook
from langchain_openai import ChatOpenAI
import os
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

llm = ChatOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    # api_key=os.getenv("OPENAI_API_KEY"),
    model="nvidia/nemotron-3.5-lightning-30b-a3b",
)

# Chapter 2.2 - Chain with a Custom Runnable

**Problem:** the first LLM returns a plain *string*, but the second prompt expects a *dict* (`{"text": ...}`).

**Solution:** wrap a normal Python function in `RunnableLambda` so it can sit inside the chain and
convert the data between two steps. Any function can become a chain step this way.

Flow: `prompt -> llm -> str -> dict -> LinkedIn prompt -> llm -> str`

In [2]:
#Task-1[Prompt]
# First prompt: answers the user's question
prompt_template = ChatPromptTemplate.from_messages([
    ("system","you are a helpful assistant"),
    ("user","{input}")
])

In [3]:
#Task-2[LLM]
# (already defined at the top, redefined so each task cell is self-contained)
llm = ChatOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    # api_key=os.getenv("OPENAI_API_KEY"),
    model="nvidia/nemotron-3.5-lightning-30b-a3b",
)

In [4]:
#Task-3[Str parser]
# AIMessage -> plain string
str_parser = StrOutputParser()

In [5]:
#Task-4[Custom Runnable]
# Adapter step: turns the string from the parser into {"text": <string>}
# so it matches the {text} variable of the next prompt.
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text:str)->dict:
    return {"text":text}

# RunnableLambda makes a plain function usable with `|`
dictionary_maker_runnable = RunnableLambda(dictionary_maker)


In [7]:
#Task-5[Template for post]
# Second prompt: takes the first answer ({text}) and turns it into a LinkedIn post
prompt_post = ChatPromptTemplate.from_messages(
    messages=[
        ("system","You're a social media post generator."),
        ("human","Create a post for the following text for LinkedIn: {text}")
    ]
)

In [9]:
#chain
# Step by step:
# prompt_template -> llm -> str_parser          : answer the question (string)
# dictionary_maker_runnable                     : string -> {"text": string}
# prompt_post -> llm -> str_parser              : write a LinkedIn post from that answer
chain = prompt_template | llm | str_parser | dictionary_maker_runnable | prompt_post | llm | str_parser

chain.invoke("What is the capital of France?")

'Paris, the City of Light, has long been a global epicenter for art, fashion, gastronomy—and yes, business innovation. 🇫🇷\n\nDid you know the capital of France is Paris? Beyond the landmarks, it’s a reminder that every city has its own “capital” of ideas, strategy, and influence.\n\nWhat’s a city that’s shaped your professional perspective? Share below! 👇\n\n#GlobalBusiness #Leadership #InternationalMindset #Paris #TravelAndWork'